In [1]:
pip install pandas numpy scikit-learn xgboost catboost matplotlib seaborn openpyxl

  Using cached pandas-3.0.2-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached scikit_learn-1.8.0-cp314-cp314-win_amd64.whl.metadata (11 kB)
  Using cached xgboost-3.2.0-py3-none-win_amd64.whl.metadata (2.1 kB)
  Using cached catboost-1.2.10-cp314-cp314-win_amd64.whl.metadata (1.5 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
  Using cached plotly-6.7.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
  Using cached narwhals-2.20.0-py3-none-any.whl.metadata (15 kB)
Using cached pandas-3.0.2-cp314-cp314-win_amd64.whl (9.9 MB)
Using cached scikit_learn-1.8.0-cp314-cp314-win_amd64.whl (8.1 MB)
Using cached xgboost-3.2.0-py3-none-win_amd64.whl (101.7 MB)
Using cached catboost-1.2.10-cp314-cp314-win_amd64.whl (101.7 MB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Using 

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\shett\\AppData\\Local\\Python\\pythoncore-3.14-64\\Lib\\site-packages\\plotly\\graph_objs\\parcats\\line\\colorbar\\title\\_font.py'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\shett\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [6]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, precision_score, recall_score, f1_score

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

import matplotlib.pyplot as plt
import seaborn as sns

# =========================
# LOAD DATA
# =========================
df = pd.read_excel(r"C:\Users\shett\Downloads\cc_teleco_proj\Telco_customer_churn.xlsx")

# =========================
# CLEAN COLUMN NAMES
# =========================
df.columns = df.columns.str.strip()

# =========================
# DROP USELESS COLUMNS
# =========================
drop_cols = [
    "Churn Label",      # duplicate of target
    "Churn Score",      # leakage
    "CLTV",             # leakage
    "Churn Reason"      # leakage (VERY IMPORTANT)
]

df.drop(columns=drop_cols, inplace=True, errors="ignore")

# =========================
# TARGET
# =========================
y = df["Churn Value"]
X = df.drop("Churn Value", axis=1)

# =========================
# HANDLE NUMERIC
# =========================
X["Total Charges"] = pd.to_numeric(X["Total Charges"], errors="coerce")
X.dropna(inplace=True)
y = y.loc[X.index]

# =========================
# ENCODE CATEGORICAL
# =========================
cat_cols = X.select_dtypes(include="object").columns

le = LabelEncoder()
for col in cat_cols:
    X[col] = le.fit_transform(X[col])

# =========================
# SCALE NUMERIC
# =========================
scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# =========================
# SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# =========================
# MODELS
# =========================
models = {
    "RF": RandomForestClassifier(n_estimators=200),
    "XGB": XGBClassifier(eval_metric='logloss'),
    "CAT": CatBoostClassifier(verbose=0)
}

# =========================
# TRAIN + EVAL
# =========================
for name, model in models.items():
    model.fit(X_train, y_train)
    
    y_prob = model.predict_proba(X_test)[:,1]
    y_pred = (y_prob > 0.3).astype(int)
    
    print(f"\n===== {name} =====")
    print(classification_report(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, y_prob))

C:\Users\shett\AppData\Local\Temp\ipykernel_19484\2414172655.py:53: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include="object").columns



===== RF =====
              precision    recall  f1-score   support

           0       0.90      0.74      0.81      1033
           1       0.51      0.76      0.61       374

    accuracy                           0.74      1407
   macro avg       0.70      0.75      0.71      1407
weighted avg       0.79      0.74      0.76      1407

ROC-AUC: 0.8395359551899613

===== XGB =====
              precision    recall  f1-score   support

           0       0.89      0.78      0.83      1033
           1       0.55      0.72      0.62       374

    accuracy                           0.77      1407
   macro avg       0.72      0.75      0.73      1407
weighted avg       0.80      0.77      0.78      1407

ROC-AUC: 0.8277769437441438

===== CAT =====
              precision    recall  f1-score   support

           0       0.91      0.76      0.83      1033
           1       0.54      0.78      0.64       374

    accuracy                           0.77      1407
   macro avg       0.7